# Standardised Data Cleaning Notebook

This notebook **standardises** the **data cleaning** process for **historical stablecoin datasets**. Users should **update** the file paths for both *data loading and saving* accordingly. The cleaned data is saved in **Parquet** format to *preserve data types*. This workflow can be easily replicated across other stablecoin datasets by modifying the file paths.

In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Data Cleaning

In [ ]:
# load data
path = "raw_data/usdt_hist_data.csv"
df = pd.read_csv(path)

In [40]:
# parse datetimes + numerics
time_cols = ["timestamp", "timeOpen", "timeClose", "timeHigh", "timeLow"]
for c in time_cols:
    df[c] = pd.to_datetime(df[c])

num_cols = ["open", "high", "low", "close", "volume", "marketCap", "circulatingSupply"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c])

df.dtypes

symbol                       object
timestamp            datetime64[ns]
timeOpen             datetime64[ns]
timeClose            datetime64[ns]
timeHigh             datetime64[ns]
timeLow              datetime64[ns]
open                        float64
high                        float64
low                         float64
close                       float64
volume                      float64
marketCap                   float64
circulatingSupply           float64
dtype: object

In [41]:
# drop duplicates (same timestamp)
print(f"Number of rows before dropping duplicates: {len(df)}")
df = df.sort_values("timestamp")
df = df.drop_duplicates(subset=["timestamp"], keep="last")
print(f"Number of rows after dropping duplicates: {len(df)}")

Number of rows before dropping duplicates: 1941
Number of rows after dropping duplicates: 1941


### Basic Price Sanity Checks

In [42]:
# ensure low > 0, high > 0, and low <= high for all observations
# if any violations exist, they should be cleaned before analysis

invalid_low = (df["low"] <= 0).sum()
invalid_high = (df["high"] <= 0).sum()
invalid_order = (df["low"] > df["high"]).sum()

print(invalid_low, invalid_high, invalid_order)

# in this dataset, all values are valid (all counts = 0),
# so no additional cleaning is required for low/high prices.

0 0 0


## Depeg Labelling

In [43]:
# create current depeg label
df["depeg"] = (abs(df["close"] - 1) >= 0.01).astype(int)
df.head()

,symbol,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply,depeg
0,USDT,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 16:35:12,2020-11-25 02:01:08,0.999805,1.001169,0.999456,1.000053,8.688931e+10,1.865547e+10,1.865448e+10,0
1,USDT,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 18:56:08,2020-11-26 08:28:09,1.000021,1.002408,0.999624,1.002035,1.146168e+11,1.899623e+10,1.895766e+10,0
2,USDT,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 01:30:10,2020-11-27 15:30:13,1.001955,1.002112,1.000664,1.001200,7.101232e+10,1.898040e+10,1.895766e+10,0
3,USDT,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 02:11:08,2020-11-28 16:13:08,1.001181,1.001583,1.000898,1.001001,5.962222e+10,1.905734e+10,1.903829e+10,0
4,USDT,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 06:08:08,2020-11-29 20:50:12,1.000981,1.001338,1.000599,1.000890,5.628394e+10,1.912689e+10,1.910989e+10,0


In [44]:
# create future depeg labels (within next h days)
horizons = [1, 3, 5, 7, 14, 30]

for h in horizons:
    df[f"depeg_future_{h}d"] = (
        df["depeg"]
        .shift(-1) # start from tomorrow
        .rolling(window=h, min_periods=1)
        .max()
        .astype("float") # keep NaN at tail first
    )

# handle tail NaNs (no future data to check, so assume no depeg)
for h in horizons:
    col = f"depeg_future_{h}d"
    df[col] = df[col].fillna(0).astype(int)

df.head()

,symbol,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d
0,USDT,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 16:35:12,2020-11-25 02:01:08,0.999805,1.001169,0.999456,1.000053,8.688931e+10,1.865547e+10,1.865448e+10,0,0,0,0,0,0,0
1,USDT,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 18:56:08,2020-11-26 08:28:09,1.000021,1.002408,0.999624,1.002035,1.146168e+11,1.899623e+10,1.895766e+10,0,0,0,0,0,0,0
2,USDT,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 01:30:10,2020-11-27 15:30:13,1.001955,1.002112,1.000664,1.001200,7.101232e+10,1.898040e+10,1.895766e+10,0,0,0,0,0,0,0
3,USDT,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 02:11:08,2020-11-28 16:13:08,1.001181,1.001583,1.000898,1.001001,5.962222e+10,1.905734e+10,1.903829e+10,0,0,0,0,0,0,0
4,USDT,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 06:08:08,2020-11-29 20:50:12,1.000981,1.001338,1.000599,1.000890,5.628394e+10,1.912689e+10,1.910989e+10,0,0,0,0,0,0,0


In [45]:
# reorder columns (place labels after first column) ---
cols = df.columns.tolist()

first_col = cols[0]
label_cols = ["depeg"] + [f"depeg_future_{h}d" for h in horizons]

# remove label cols first
remaining_cols = [c for c in cols if c not in label_cols]

# rebuild column order
new_cols = [first_col] + label_cols + [c for c in remaining_cols if c != first_col]

df = df[new_cols]

df.head()

,symbol,depeg,depeg_future_1d,depeg_future_3d,depeg_future_5d,depeg_future_7d,depeg_future_14d,depeg_future_30d,timestamp,timeOpen,timeClose,timeHigh,timeLow,open,high,low,close,volume,marketCap,circulatingSupply
0,USDT,0,0,0,0,0,0,0,2020-11-25 23:59:59,2020-11-25,2020-11-25 23:59:59,2020-11-25 16:35:12,2020-11-25 02:01:08,0.999805,1.001169,0.999456,1.000053,8.688931e+10,1.865547e+10,1.865448e+10
1,USDT,0,0,0,0,0,0,0,2020-11-26 23:59:59,2020-11-26,2020-11-26 23:59:59,2020-11-26 18:56:08,2020-11-26 08:28:09,1.000021,1.002408,0.999624,1.002035,1.146168e+11,1.899623e+10,1.895766e+10
2,USDT,0,0,0,0,0,0,0,2020-11-27 23:59:59,2020-11-27,2020-11-27 23:59:59,2020-11-27 01:30:10,2020-11-27 15:30:13,1.001955,1.002112,1.000664,1.001200,7.101232e+10,1.898040e+10,1.895766e+10
3,USDT,0,0,0,0,0,0,0,2020-11-28 23:59:59,2020-11-28,2020-11-28 23:59:59,2020-11-28 02:11:08,2020-11-28 16:13:08,1.001181,1.001583,1.000898,1.001001,5.962222e+10,1.905734e+10,1.903829e+10
4,USDT,0,0,0,0,0,0,0,2020-11-29 23:59:59,2020-11-29,2020-11-29 23:59:59,2020-11-29 06:08:08,2020-11-29 20:50:12,1.000981,1.001338,1.000599,1.000890,5.628394e+10,1.912689e+10,1.910989e+10


---

## Save

In [46]:
df.dtypes

symbol                       object
depeg                         int64
depeg_future_1d               int64
depeg_future_3d               int64
depeg_future_5d               int64
depeg_future_7d               int64
depeg_future_14d              int64
depeg_future_30d              int64
timestamp            datetime64[ns]
timeOpen             datetime64[ns]
timeClose            datetime64[ns]
timeHigh             datetime64[ns]
timeLow              datetime64[ns]
open                        float64
high                        float64
low                         float64
close                       float64
volume                      float64
marketCap                   float64
circulatingSupply           float64
dtype: object

In [47]:
df.to_parquet("clean_data/usdt_clean.parquet")